In [7]:
import pandas as pd

# Read the file
df = pd.read_csv("demand_ABU_50K_kwh_15minIntervals.csv")

# Parse timestamp column
df["Timestamp"] = pd.to_datetime(df["Timestamp"], format="%d.%m.%Y %H:%M")

# Set timestamp as index
df = df.set_index("Timestamp")

# Convert 15-minute kWh values to hourly kWh by summing every 4 intervals
hourly_demand = df[["kwh"]].resample("h").sum()

# Optional: rename column
hourly_demand = hourly_demand.rename(columns={"kwh": "kwh_hourly"})

# Save to a new file
hourly_demand.to_csv("demand_ABU_50K_kwh_hourly.csv")

In [8]:
# Preview
print(hourly_demand.head(20))

                       kwh_hourly
Timestamp                        
2025-01-01 00:00:00  1.043162e+16
2025-01-01 01:00:00  1.062264e+16
2025-01-01 02:00:00  1.081367e+16
2025-01-01 03:00:00  9.285442e+15
2025-01-01 04:00:00  5.655894e+15
2025-01-01 05:00:00  1.234190e+16
2025-01-01 06:00:00  2.323055e+16
2025-01-01 07:00:00  3.959126e+15
2025-01-01 08:00:00  1.378288e+15
2025-01-01 09:00:00  1.317591e+14
2025-01-01 10:00:00  3.693846e+14
2025-01-01 11:00:00  1.993689e+15
2025-01-01 12:00:00  3.658355e+14
2025-01-01 13:00:00  1.415774e+15
2025-01-01 14:00:00  3.390868e+15
2025-01-01 15:00:00  1.637511e+14
2025-01-01 16:00:00  1.679106e+15
2025-01-01 17:00:00  1.372739e+15
2025-01-01 18:00:00  2.262358e+15
2025-01-01 19:00:00  3.232496e+16


In [10]:
# -------------------------
# 1. Load base hourly demand
# -------------------------
base = pd.read_csv("demand_ABU_50K_kwh_hourly.csv")

base["Timestamp"] = pd.to_datetime(base["Timestamp"])
base = base.set_index("Timestamp")

# If needed, apply scaling here
# base["kwh_hourly"] = base["kwh_hourly"] / 1e15

# -------------------------
# 2. Load EV demand
# -------------------------
ev = pd.read_csv("EMobility_Demand.csv")

# If first row is a unit row, remove it
ev = ev.iloc[1:].copy()

# Rename columns
ev.columns = ["Time", "ev_kwh"]

# Add year manually
ev["Timestamp"] = pd.to_datetime(
    ev["Time"] + "2025",
    format="%d.%m. %H:%M%Y"
)

ev["ev_kwh"] = pd.to_numeric(ev["ev_kwh"], errors="coerce")
ev = ev.set_index("Timestamp")

# Keep only needed column
ev = ev[["ev_kwh"]]

# -------------------------
# 3. Join both datasets
# -------------------------
combined = base.join(ev, how="left")
combined["ev_kwh"] = combined["ev_kwh"].fillna(0)

# -------------------------
# 4. Create final hourly demand
# -------------------------
combined["total_kwh"] = combined["kwh_hourly"] + combined["ev_kwh"]

# -------------------------
# 5. Save result
# -------------------------
combined.to_csv("final_hourly_demand.csv")

print(combined.head(20))
print(combined[["kwh_hourly", "ev_kwh", "total_kwh"]].tail())

                       kwh_hourly    ev_kwh     total_kwh
Timestamp                                                
2025-01-01 00:00:00  1.040000e+16   0.00000  1.040000e+16
2025-01-01 01:00:00  1.060000e+16   0.00000  1.060000e+16
2025-01-01 02:00:00  1.080000e+16   0.00000  1.080000e+16
2025-01-01 03:00:00  9.285440e+15   0.00000  9.285440e+15
2025-01-01 04:00:00  5.655890e+15   0.00000  5.655890e+15
2025-01-01 05:00:00  1.230000e+16   0.00000  1.230000e+16
2025-01-01 06:00:00  2.320000e+16   0.00000  2.320000e+16
2025-01-01 07:00:00  3.959130e+15   0.00000  3.959130e+15
2025-01-01 08:00:00  1.378290e+15   0.00000  1.378290e+15
2025-01-01 09:00:00  1.317590e+14   6.89630  1.317590e+14
2025-01-01 10:00:00  3.693850e+14  49.56600  3.693850e+14
2025-01-01 11:00:00  1.993690e+15  32.43082  1.993690e+15
2025-01-01 12:00:00  3.658360e+14  21.83500  3.658360e+14
2025-01-01 13:00:00  1.415770e+15   0.00000  1.415770e+15
2025-01-01 14:00:00  3.390870e+15   0.00000  3.390870e+15
2025-01-01 15: